# Suite CTX — Contexts: the context a procedure takes, as schema data

A context is the standardized argument of a procedure, validation among them, held as schema data
([CAP-319: contexts as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-319)):
a `Contexts.OfContext` value holds `variables`, `symbols` and `parameters`, each keyed by name. A `Contexts.OfVariable`
gives a variable its schema and its value; a `Contexts.OfSymbol` gives a symbol an object schema and the object it
binds; a `Contexts.OfParameter` gives a parameter its schema and either the object it is bound to or the value
assigned to it. Each refers to what it holds by reference: a schema by a `References.OfSchema`, an object by a
`References.OfObject`, a value by a `References.OfValue`
([CAP-320: schema and value references](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-320)).
A sub-context is copied: `Contexts.within(store, outer, own)` gives a new context that holds the outer context's
entries, each entry of its own replacing the outer entry of its name, and nothing links it to either.
`Contexts.validate(store, context)` reports a context's problems, a symbol whose schema is not an object schema among
them. Validation reads a context's parameters (suite PAR, PAR-09..12).

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store are `Proxy` objects
typed loosely, so callbacks are annotated `(t: any)`; plain mappings are `Map`s, ints `bigint`s.

In [ ]:
import { Builders as B, Contexts, JSON as Json, Paths, Plain, References, Validators } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

/** pcb.Chip, a singleton; pcb.Board, which feeds chips; pcb.Part, a union of the two. */
function parts(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("ctx.parts/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Chip: S.OfObject, Board: S.OfObject, Feeds: S.OfRelation, Part: S.OfUnion })
    .spec([
      (_: any) => _.Feeds.name("pcb.Feeds").links([(t: any) => t.name("board"), (t: any) => t.name("chip")]),
      (_: any) => _.Chip.name("pcb.Chip").ref(true).singleton("pcb.TheChip").adjacencies([
        (a: any) => a.relation([_.Feeds]).me("chip")]),
      (_: any) => _.Board.name("pcb.Board").ref(true).properties({ size: S.OfInt }).adjacencies({
        feeds: (a: any) => a.relation([_.Feeds]).me("board") }),
      (_: any) => _.Part.name("pcb.Part").branches([(t: any) => t.name("chip").schema([_.Chip]),
        (t: any) => t.name("board").schema([_.Board])]),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(parts);
const [Chip, Board, Part, Int] = ["pcb.Chip", "pcb.Board", "pcb.Part", "Schemas.OfInt"].map((name) => store.registered(name));
const OfContext = store.registered("Contexts.OfContext");

/** A schema object's reference (`References.OfSchema`) to `schema`, through the builders. */
function at(schema: unknown): (r: any) => any {
  return (r: any) => r.path(Paths.of(schema));
}

function plain(value: any): any {
  return Plain.ToPlain(store)(value.as_schema(), value);
}

/** A schema object's path, from the root module through each module and the schema, by name. */
function schemaPath(...names: string[]): unknown {
  let nested: unknown = { target: {} };
  for (const name of [...names].reverse()) nested = { entry: { i: "schemas", o: "", properties: { name }, next: nested } };
  return { path: { singleton: ".", next: nested } };
}

<a id="CTX-01"></a>
## CTX-01 · Every store holds the context schemas, its own, value objects: a context's variables, symbols and parameters, each keyed by name and each a name, a schema object's reference and what it holds by reference; Contexts.install, run again, writes nothing

In [ ]:
const [OfVariable, OfSymbol, OfParameter] = ["OfVariable", "OfSymbol", "OfParameter"].map((name) => store.registered(`Contexts.${name}`));
const [OfSchema, OfObject, OfValue] = ["OfSchema", "OfObject", "OfValue"].map((name) => store.registered(`References.${name}`));
assert(equal([...OfVariable.properties.keys()], ["name", "schema", "value"]) && equal([...OfSymbol.properties.keys()], ["name", "schema", "object"]));
assert(equal([...OfParameter.properties.keys()], ["name", "schema", "object", "value"]));
assert(same(["schema", "value"].map((p) => OfVariable.properties.get(p).schema), [OfSchema, OfValue]));
assert(same(["schema", "object", "value"].map((p) => OfParameter.properties.get(p).schema), [OfSchema, OfObject, OfValue]));
assert(OfSymbol.properties.get("object").schema === OfObject);
assert(equal([...OfContext.properties.keys()], ["variables", "symbols", "parameters"]));
for (const [name, element] of [["variables", OfVariable], ["symbols", OfSymbol], ["parameters", OfParameter]] as [string, unknown][]) {
  const container = OfContext.properties.get(name).schema; // keyed by name
  assert(container.index === store.Schemas.OfStr.as_schema() && container.element === element);
}
assert(![OfVariable, OfSymbol, OfParameter, OfContext].some((schema) => schema.ref)); // value objects
assert((new B.Store() as any).registered("Contexts.OfContext") !== OfContext); // each store's its own
{
  const names = new Map([...store.names()].map((name: string) => [name, store.registered(name)]));
  store.transaction(Contexts.install); // each found by its name
  const after = new Map([...store.names()].map((name: string) => [name, store.registered(name)]));
  assert(after.size === names.size && [...names].every(([name, schema]) => after.get(name) === schema));
}

<a id="CTX-02"></a>
## CTX-02 · A context is built through the builders' fluent calls, written as JSON and read back the same; its entries are read by name, each as its visitor

In [ ]:
store.transaction(() => {
  const chip = References.of(store, store.singleton("pcb.TheChip"));
  const context = store.Contexts.OfContext((c: any) => c
    .variables({ x: (v: any) => v.name("x").schema(at(Int)).value((r: any) => r.value(3n)) })
    .symbols({ c: (s: any) => s.name("c").schema(at(Chip)).object(chip) })
    .parameters({ n: (p: any) => p.name("n").schema(at(Int)).value((r: any) => r.value(4n)),
      h: (p: any) => p.name("h").schema(at(Chip)).object(chip) }));
  const written = map({
    variables: { x: { name: "x", schema: schemaPath("Schemas", "OfInt"), value: { value: { int: 3n } } } },
    symbols: { c: { name: "c", schema: schemaPath("pcb", "Chip"), object: plain(chip) } },
    parameters: { n: { name: "n", schema: schemaPath("Schemas", "OfInt"), value: { value: { int: 4n } } },
      h: { name: "h", schema: schemaPath("pcb", "Chip"), object: plain(chip) } } });
  assert(equal(plain(context), written)); // a value, written as any value is
  const text = Json.ToJSON(store)(OfContext, context);
  assert(equal(plain(Json.FromJSON(store)(OfContext, text)), written)); // and read back the same
  assert(equal([...Contexts.entries(store, context, "parameters").keys()], ["n", "h"])); // its entries, by name, as visitors
  assert(Contexts.held(Contexts.entries(store, context, "parameters").get("h").value) === null); // a property not set
  assert(same(Contexts.validate(store, context), []));
});

<a id="CTX-03"></a>
## CTX-03 · A sub-context is copied: `Contexts.within` gives a new context holding the outer context's entries, each entry of its own replacing the outer one of its name, the outer context unchanged; validation reads each context's own parameters; what is not a context is refused

In [ ]:
store.transaction(() => {
  const chip = References.of(store, store.singleton("pcb.TheChip"));
  const outer = store.Contexts.OfContext((c: any) => c
    .symbols({ c: (s: any) => s.name("c").object(chip) })
    .parameters({ n: (p: any) => p.name("n").value((r: any) => r.value(3n)), h: (p: any) => p.name("h").object(chip) }));
  const own = store.Contexts.OfContext((c: any) => c.parameters({ // what a statement assigns: n anew, and k
    n: (p: any) => p.name("n").value((r: any) => r.value("x")), k: (p: any) => p.name("k").value((r: any) => r.value(5n)) }));
  const before = plain(outer);
  const sub = Contexts.within(store, outer, own);
  assert(equal(plain(sub), map({ symbols: before.get("symbols"), parameters: {
    n: { name: "n", value: { value: { str: "x" } } }, h: before.get("parameters").get("h"),
    k: { name: "k", value: { value: { int: 5n } } } } }))); // the outer entries, each of its own replacing one
  const empty = () => store.Contexts.OfContext((c: any) => c);
  assert(equal(plain(outer), before) && equal(plain(Contexts.within(store, outer, empty())), before));
  assert(equal(plain(Contexts.within(store, empty(), empty())), new Map()));
  const n = store.References.OfParameter((p: any) => p.name("n"));
  assert(same(Validators.Validate(store).OfScalar(Int, n, outer), [])); // validation reads each context's own parameters
  assert(same(Validators.Validate(store).OfScalar(Int, n, sub), [
    "the value: parameter 'n' is assigned a value of another schema than its place expects"]));
  for (const wrong of [new Map([["n", 3n]]), n]) {
    raises(TypeError, () => Contexts.within(store, outer, wrong), "Contexts.within takes a context (Contexts.OfContext), not ");
  }
});

<a id="CTX-04"></a>
## CTX-04 · `Contexts.validate` reports an entry named otherwise than its key, a symbol whose schema is not an object schema, a schema reference that designates no schema object the store holds or whose path is not a schema object's, and a parameter both bound to an object and assigned a value; what is not a context is refused

In [ ]:
store.transaction(() => {
  const chip = References.of(store, store.singleton("pcb.TheChip"));
  const nowhere = (r: any) => r.path((p: any) => p.singleton(".").next((x: any) => x.entry((e: any) => e.i("schemas").o("")
    .properties({ name: "nowhere" }).next((y: any) => y.target((t: any) => t)))));
  const context = store.Contexts.OfContext((c: any) => c
    .variables({ x: (v: any) => v.name("y").schema(at(Part)) }) // any schema, a union among them
    .symbols({ u: (s: any) => s.name("u").schema(at(Part)).object(chip), // an object schema's alone
      c: (s: any) => s.name("c").schema(at(Chip)) })
    .parameters({ n: (p: any) => p.name("n").schema(nowhere),
      b: (p: any) => p.name("b").object(chip).value((r: any) => r.value(3n)),
      s: (p: any) => p.name("s").schema((r: any) => r.path(Paths.of(store.singleton("pcb.TheChip")))) }));
  assert(same(Contexts.validate(store, context), [
    "variable 'x': its entry is named 'y'",
    "symbol 'u': a schema object's reference designates a 'Schemas.OfUnion', where a 'Schemas.OfObject' is expected",
    "parameter 'n': a schema object's reference designates no schema object the store holds",
    "parameter 'b' is both bound to an object and assigned a value",
    "parameter 's': a schema object's reference (References.OfSchema): a schema object's path starts from the root "
    + "module, '.', and each step names an entry, {'i': adjacency, 'o': '', 'properties': {'name': name}}"]));
  raises(TypeError, () => Contexts.validate(store, chip), "Contexts.entries takes a context (Contexts.OfContext), not ");
});